All the following is based on the best model before reweighting the cost functions:

In [ ]:
# SHAP:

import shap
import numpy as np
import pandas as pd

# explainer
explainer_1 = shap.TreeExplainer(pipe.named_steps["model_1"])  # for Y>=2
explainer_2 = shap.TreeExplainer(pipe.named_steps["model_2"])  # for Y>=3
# SHAP values
shap_values_1 = explainer_1.shap_values(X_test)
shap_values_2 = explainer_2.shap_values(X_test)
# =========================
# FIX shape
# =========================
if isinstance(shap_values_1, list):
    shap_abs_1 = np.sum([np.abs(sv) for sv in shap_values_1], axis=0)

else:
    shap_abs_1 = np.abs(shap_values_1)

if isinstance(shap_values_2, list):
    shap_abs_2 = np.sum([np.abs(sv) for sv in shap_values_2], axis=0)

else:
    shap_abs_2 = np.abs(shap_values_2)

# global SHAP
global_shap_1 = np.mean(shap_abs_1, axis=0)
global_shap_2 = np.mean(shap_abs_2, axis=0)

# dataframe
feature_importance_1 = pd.DataFrame({
    "feature": X_test.columns,
    "global_shap_1": global_shap_1
}).sort_values(by="global_shap", ascending=False)

feature_importance_2 = pd.DataFrame({
    "feature": X_test.columns,
    "global_shap_2": global_shap_2
}).sort_values(by="global_shap", ascending=False)

print(feature_importance_1.head(20))
print(feature_importance_2.head(20))


GAM:  

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pygam import LogisticGAM, s, f
import matplotlib.pyplot as plt


X = pd.read_csv("all_x.csv")
y = pd.read_csv("all_y.csv")
df = X.merge(y, on="building_id")

X = df.drop(["building_id", "damage_grade"], axis=1)
y = df["damage_grade"] - 1   # change the namely level 1,2,3 into 0,1,2


# training-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# drop has_secondary_use to avoid double counting
X_train_clean = X_train.drop(columns=["has_secondary_use"], errors="ignore")
X_test_clean  = X_test.drop(columns=["has_secondary_use"], errors="ignore")



In [23]:

y_train_2 = (y_train >= 1).astype(int)
y_train_3 = (y_train >= 2).astype(int)


# =========================
# 2. one-hot（必须）
# =========================
X_gam = pd.get_dummies(X_train_clean, drop_first=False)


# =========================
# 3. 判断categorical
# =========================
def is_categorical(col):
    return (
        set(X_gam[col].unique()) <= {0,1}
    )


# to construct GAM terms:
terms = None

for i, col in enumerate(X_gam.columns):
    term = s(i) 

    if terms is None:
        terms = term
    else:
        terms = terms + term   


# =========================
# 5. 训练模型
# =========================
gam_2 = LogisticGAM(terms).fit(X_gam.values, y_train_2.values)
gam_3 = LogisticGAM(terms).fit(X_gam.values, y_train_3.values)


ValueError: Inconsistent input and output data shapes. found X: (208480, 67) and y: (416960,)